# Neural chunking walkthrough

This notebook shows the retained BiLSTM run and the code behind it. Each token receives a BIO tag: `B-X` starts a chunk of type `X`, `I-X` continues it, and `O` means the token is outside a chunk.

The pipeline also implements a compact Transformer encoder. The committed evidence here is for the BiLSTM only. This notebook reads committed files, does not train a model, and does not download a corpus.


In [1]:
import json
from pathlib import Path

_candidates = [Path.cwd(), Path.cwd() / "projects" / "neural-chunking", Path.cwd().parent]
PROJECT_ROOT = next(
    candidate
    for candidate in _candidates
    if (candidate / "artifacts" / "results" / "retained_metrics.json").exists()
)
metrics_path = PROJECT_ROOT / "artifacts" / "results" / "retained_metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
figure_paths = [
    PROJECT_ROOT / "artifacts" / "figures" / "bilstm_accuracy_curve.png",
    PROJECT_ROOT / "artifacts" / "figures" / "bilstm_loss_curve.png",
    PROJECT_ROOT / "artifacts" / "figures" / "best_bilstm_confusion_matrix.png",
]
assert all(path.is_file() for path in figure_paths)
print(f"Loaded {metrics_path.relative_to(PROJECT_ROOT)}")
print(f"Status: {metrics['status']}")

Loaded artifacts/results/retained_metrics.json
Status: retained_run


## Results

The BiLSTM checkpoint was selected at epoch 16 using validation token macro F1, with validation loss as the tie-breaker.


In [2]:
result_rows = [
    ("Token accuracy", metrics["test"]["token_accuracy"]),
    ("Token macro F1", metrics["test"]["token_macro_f1"]),
    ("Token weighted F1", metrics["test"]["token_weighted_f1"]),
    ("Matthews correlation", metrics["test"]["token_matthews_correlation"]),
    ("Test loss", metrics["test"]["loss"]),
]
print("held-out token metric\tvalue")
for label, value in result_rows:
    print(f"{label}\t{value:.4f}")

held-out token metric	value
Token accuracy	0.9430
Token macro F1	0.7521
Token weighted F1	0.9423
Matthews correlation	0.9288
Test loss	0.2158


The split contains 6,243 training sentences, 1,337 validation sentences, and 1,339 test sentences (31,833 test tokens). Exact-span F1 is unavailable because the retained predictions were not kept. The evaluator calculates it for new runs, but it is not reconstructed here.


## Training behaviour

These committed figures show the selected BiLSTM learning curves and held-out confusion matrix.

![BiLSTM token accuracy by epoch](../artifacts/figures/bilstm_accuracy_curve.png)

![BiLSTM loss by epoch](../artifacts/figures/bilstm_loss_curve.png)

![BiLSTM confusion matrix](../artifacts/figures/best_bilstm_confusion_matrix.png)


## What the implementation demonstrates

- **Data handling:** [`data.py`](../src/neural_chunking/data.py) parses sentences, builds vocabularies from training data, and assigns complete sentences to splits with a seeded content hash. Exact duplicate token sequences stay together.
- **Padding control:** `collate_sentences` returns a valid-token mask. The BiLSTM packs real tokens only; the Transformer masks padding in self-attention; loss and metrics ignore padded positions.
- **Model design:** [`models.py`](../src/neural_chunking/models.py) provides `BiLSTMChunker` and `TransformerChunker` behind the same token-classification interface.
- **Metrics:** [`metrics.py`](../src/neural_chunking/metrics.py) converts BIO tags to exact spans and aggregates precision, recall, and F1 alongside token metrics.


## Evaluation boundary and code map

[`training.py`](../src/neural_chunking/training.py) selects a checkpoint on validation span F1, freezes that selection, then evaluates the test partition once. Checkpoint and input digests are recorded for verification. [`cli.py`](../src/neural_chunking/cli.py) exposes separate train and evaluate commands.

The raw corpus is not committed. The retained run does not include its corpus identity or sentence-level predictions, so the notebook reports only metrics present in [`retained_metrics.json`](../artifacts/results/retained_metrics.json).


## Scope

No Transformer score is claimed because none is present in the committed evidence. No exact-span F1 is inferred from token metrics. The project can produce both measures when a new, reproducible input and checkpoint are supplied.
